<a href="https://colab.research.google.com/github/thinkblue9/korean-exam-review-2026-05/blob/main/Colab_%EC%8B%9C%EC%9E%91%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===== 셀 1: GPU 확인 =====
# 런타임 > 런타임 유형 변경 > 하드웨어 가속기를 'T4 GPU'로 먼저 설정하세요.
import torch
print("CUDA 사용 가능:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 — 런타임 유형을 T4 GPU로 바꾸세요")

CUDA 사용 가능: True
GPU: Tesla T4


In [2]:
# ===== 셀 2: marker 설치 =====
!pip install -q marker-pdf
print("설치 완료")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796

In [ ]:
# ===== 셀 3: 구글 드라이브 마운트 =====
# 드라이브에 'marker_input' 폴더를 만들고 그 안에 7개 대학 PDF를 올려두세요.
from google.colab import drive
drive.mount('/content/drive')

import os
INPUT_DIR  = '/content/drive/MyDrive/marker_input'    # PDF 넣어둔 폴더
OUTPUT_DIR = '/content/drive/MyDrive/marker_output'   # 결과 저장될 폴더
os.makedirs(OUTPUT_DIR, exist_ok=True)

pdfs = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith('.pdf')]
print(f"입력 PDF {len(pdfs)}개:")
for p in pdfs:
    print("  -", p)

In [ ]:
# ===== 셀 4: 변환 실행 (PDF 1개씩 순차 처리) =====
# 1개씩 돌리는 이유: 중간에 끊겨도 끝난 파일은 보존되고, 어느 파일이 문제인지 바로 보임.
import subprocess, time

for i, pdf in enumerate(pdfs, 1):
    src = os.path.join(INPUT_DIR, pdf)
    print(f"\n===== [{i}/{len(pdfs)}] {pdf} 변환 시작 =====")
    t0 = time.time()
    result = subprocess.run([
        "marker_single", src,
        "--output_dir", OUTPUT_DIR,
        "--output_format", "markdown",
        # force_ocr 미지정 → 텍스트 레이어 그대로 사용 (표 인식 모델은 항상 동작)
    ], capture_output=True, text=True)
    print(result.stdout[-2000:])      # 진행 로그 일부
    if result.returncode != 0:
        print("⚠️ 오류 발생:")
        print(result.stderr[-2000:])
    print(f"===== {pdf} 완료, {time.time()-t0:.0f}초 =====")

print("\n전체 변환 종료")

In [ ]:
# ===== 셀 5: 결과 zip으로 묶어 다운로드 =====
import shutil
from google.colab import files
shutil.make_archive('/content/marker_result', 'zip', OUTPUT_DIR)
files.download('/content/marker_result.zip')